# vLLM weight-sync demo (no TRL)

Pairs with `script/vllm_serve_minimal.py`. Start the server first in a terminal:

```bash
CUDA_VISIBLE_DEVICES=0 NCCL_CUMEM_ENABLE=0 \
    python script/vllm_serve_minimal.py \
        --model Qwen/Qwen3-0.6B --port 8000 --enforce-eager
```

This notebook then connects from a *different* GPU, generates, mutates the local copy of the model, broadcasts the new weights into the running vLLM workers, and generates again to confirm the change took effect.

In [ ]:
import socket
import requests
import torch
from transformers import AutoModelForCausalLM

BASE       = "http://127.0.0.1:8000"
MODEL_NAME = "Qwen/Qwen3-0.6B"
TRAIN_DEV  = "cuda:1"   # MUST be a different physical GPU than the vLLM server

assert requests.get(f"{BASE}/health/").ok, "start vllm_serve_minimal.py first"

In [ ]:
def generate(prompts, max_tokens=32, temperature=0.0):
    r = requests.post(f"{BASE}/generate/", json={
        "prompts": prompts,
        "max_tokens": max_tokens,
        "temperature": temperature,
    })
    r.raise_for_status()
    return r.json()

PROMPT = "The capital of France is"
before = generate([PROMPT])
print("before:", before["completion_text"])

In [ ]:
# Load a local copy of the same model. This is what your custom GRPO
# trainer would step on; here we just perturb it for a sanity check.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16,
).to(TRAIN_DEV)
model.eval()

In [ ]:
from vllm.distributed.device_communicators.pynccl import PyNcclCommunicator
from vllm.distributed.utils import StatelessProcessGroup

def _free_port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p

PG_HOST     = "127.0.0.1"
PG_PORT     = _free_port()
WORLD_SIZE  = 2                  # 1 vLLM worker + 1 client (this notebook)
CLIENT_RANK = WORLD_SIZE - 1     # required by WeightSyncExt

client_uuid = str(torch.cuda.get_device_properties(TRAIN_DEV).uuid)

# Tell the server to spin up its NCCL side. Returns immediately; its
# ncclCommInitRank then blocks until we join below.
requests.post(f"{BASE}/init_communicator/", json={
    "host": PG_HOST, "port": PG_PORT,
    "world_size": WORLD_SIZE,
    "client_device_uuid": client_uuid,
}).raise_for_status()

pg = StatelessProcessGroup.create(
    host=PG_HOST, port=PG_PORT, rank=CLIENT_RANK, world_size=WORLD_SIZE,
)
comm = PyNcclCommunicator(pg, device=TRAIN_DEV)
print("NCCL group ready")

In [ ]:
def push_param(name, tensor):
    """Broadcast one named tensor from this process into the vLLM workers."""
    t = tensor.detach().to(TRAIN_DEV).contiguous()
    dtype_str = f"torch.{str(t.dtype).split('.')[-1]}"   # e.g. 'torch.bfloat16'
    requests.post(f"{BASE}/update_named_param/", json={
        "name": name, "dtype": dtype_str, "shape": list(t.shape),
    }).raise_for_status()
    comm.broadcast(t, src=CLIENT_RANK)
    comm.group.barrier()

def push_all_params():
    n = 0
    for name, p in model.named_parameters():
        push_param(name, p.data)
        n += 1
    return n

In [ ]:
# Sanity check: perturb local weights, push, regenerate. Output should differ.
with torch.no_grad():
    for p in model.parameters():
        p.add_(torch.randn_like(p) * 1e-3)

n_pushed = push_all_params()
print(f"pushed {n_pushed} tensors")

after = generate([PROMPT])
print("before:", before["completion_text"])
print("after :", after["completion_text"])

In [ ]:
requests.post(f"{BASE}/close_communicator/")

## Plugging this into a custom GRPO loop

1. Replace the perturbation cell with your training step (one optimizer update on `model`).
2. Call `push_all_params()` after each step (or every N steps) to sync weights into vLLM.
3. Use `generate(...)` to collect rollouts for the next GRPO batch.
4. Compute rewards locally, build advantages, and step the optimizer again.

If you later want multi-GPU training (FSDP / DeepSpeed ZeRO-3), only rank 0 should call `push_param` — gather full params first via `summon_full_params` (FSDP) or `GatheredParameters` (DeepSpeed).